# 0. Load in the cleaned test and train data

In [1]:
import pandas as pd

test = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_test_cleaned.csv")
test.info()
submission = test[["GuestID"]].copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 60 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   GuestID                     1739 non-null   int64
 1   AllInclusive                1739 non-null   int64
 2   VIP                         1739 non-null   int64
 3   RoomService                 1739 non-null   int64
 4   Dining                      1739 non-null   int64
 5   Retail                      1739 non-null   int64
 6   Spa                         1739 non-null   int64
 7   Entertainment               1739 non-null   int64
 8   LoyaltyPoints               1739 non-null   int64
 9   SurveyScore                 1739 non-null   int64
 10  DaysSinceEmail              1739 non-null   int64
 11  SharedRoom                  1739 non-null   int64
 12  RoomNumber                  1739 non-null   int64
 13  BookingMonth                1739 non-null   int64
 14  BookingY

In [2]:
test.isna().sum()

GuestID                       0
AllInclusive                  0
VIP                           0
RoomService                   0
Dining                        0
Retail                        0
Spa                           0
Entertainment                 0
LoyaltyPoints                 0
SurveyScore                   0
DaysSinceEmail                0
SharedRoom                    0
RoomNumber                    0
BookingMonth                  0
BookingYear                   0
PromoCode_PromoA              0
PromoCode_PromoB              0
Region_AsiaPacific            0
Region_Europe                 0
Region_Unknown                0
PackageType_NoPackage         0
PackageType_Relaxation        0
PackageType_Wellness          0
AgeGroup_Middle               0
AgeGroup_Minor                0
AgeGroup_Senior               0
AgeGroup_Unknown              0
AgeGroup_Young                0
RoomFloor_B                   0
RoomFloor_C                   0
RoomFloor_D                   0
RoomFloo

In [3]:
train = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_train_cleaned.csv")
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 61 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   GuestID                     6954 non-null   int64
 1   AllInclusive                6954 non-null   int64
 2   VIP                         6954 non-null   int64
 3   RoomService                 6954 non-null   int64
 4   Dining                      6954 non-null   int64
 5   Retail                      6954 non-null   int64
 6   Spa                         6954 non-null   int64
 7   Entertainment               6954 non-null   int64
 8   LoyaltyPoints               6954 non-null   int64
 9   SurveyScore                 6954 non-null   int64
 10  DaysSinceEmail              6954 non-null   int64
 11  Churned                     6954 non-null   int64
 12  SharedRoom                  6954 non-null   int64
 13  RoomNumber                  6954 non-null   int64
 14  BookingM

In [4]:
train.isna().sum()

GuestID                       0
AllInclusive                  0
VIP                           0
RoomService                   0
Dining                        0
                             ..
ReferralSource_TikTok         0
ReferralSource_TripAdvisor    0
ReferralSource_Twitter        0
ReferralSource_Yelp           0
ReferralSource_YouTube        0
Length: 61, dtype: int64

#Create X and y variables for modeling

from sklearn.model_selection import train_test_split

X = train.drop('Churned', axis=1)
y = train['Churned']

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Train the model - This is where your model goes

In [5]:
import optuna
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import StratifiedKFold, train_test_split

# GuestID is an identifier — drop from features.
drop_cols = ["GuestID", "Churned"]
X = train.drop(columns=drop_cols)
y = train["Churned"]

# XGBoost needs numeric features. Build shared category maps from train+test so the
# test-time encoding lines up with what the model was trained on.
cat_features = [c for c in X.columns if X[c].dtype == "object"]
test_features = test.drop(columns=["GuestID"]) if "GuestID" in test.columns else test.copy()

category_maps = {}
for c in cat_features:
    combined = pd.concat([X[c].astype(str), test_features[c].astype(str)]).fillna("missing")
    category_maps[c] = pd.Categorical(combined).categories

X_enc = X.copy()
for c in cat_features:
    X_enc[c] = pd.Categorical(
        X_enc[c].astype(str).fillna("missing"), categories=category_maps[c]
    ).codes

X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42, stratify=y
)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos


def objective(trial):
    params = {
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "auc",
        "early_stopping_rounds": 50,
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        preds = model.predict(X_va)
        scores.append(roc_auc_score(y_va, preds))

        trial.report(sum(scores) / len(scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best CV ROC-AUC: {study.best_value:.4f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# Refit best model on the train split, evaluate on the held-out test split.
best_params = {
    **study.best_params,
    "n_estimators": 1000,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "auc",
    "early_stopping_rounds": 50,
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}

xgb = XGBClassifier(**best_params)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = xgb.predict(X_test)
print(f"\nTest ROC-AUC:   {roc_auc_score(y_test, y_pred):.3f}")
print(f"Best iteration: {xgb.best_iteration}")
print(classification_report(y_test, y_pred))

fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb.feature_importances_,
}).sort_values("importance", ascending=False)
print(fi.to_string(index=False))

[I 2026-05-23 18:21:06,239] A new study created in memory with name: no-name-b3cd9e9d-6250-46f2-aec5-ca99909108dd


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-23 18:21:07,040] Trial 0 finished with value: 0.8201796838463208 and parameters: {'max_depth': 5, 'learning_rate': 0.22648248189516842, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'min_child_weight': 4, 'gamma': 0.7799726016810132, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.915443189153755}. Best is trial 0 with value: 0.8201796838463208.
[I 2026-05-23 18:21:08,588] Trial 1 finished with value: 0.8057768978407394 and parameters: {'max_depth': 7, 'learning_rate': 0.05675206026988745, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'min_child_weight': 17, 'gamma': 1.0616955533913808, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 0 with value: 0.8201796838463208.
[I 2026-05-23 18:21:12,362] Trial 2 finished with value: 0.8146089786792787 and parameters: {'max_depth': 5, 'learning_rate': 0.0199473547030745, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 

# 2. Test - This will produce the test data

In [6]:
test.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 60 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   GuestID                     1739 non-null   int64
 1   AllInclusive                1739 non-null   int64
 2   VIP                         1739 non-null   int64
 3   RoomService                 1739 non-null   int64
 4   Dining                      1739 non-null   int64
 5   Retail                      1739 non-null   int64
 6   Spa                         1739 non-null   int64
 7   Entertainment               1739 non-null   int64
 8   LoyaltyPoints               1739 non-null   int64
 9   SurveyScore                 1739 non-null   int64
 10  DaysSinceEmail              1739 non-null   int64
 11  SharedRoom                  1739 non-null   int64
 12  RoomNumber                  1739 non-null   int64
 13  BookingMonth                1739 non-null   int64
 14  BookingY

In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 59 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   AllInclusive                6954 non-null   int64
 1   VIP                         6954 non-null   int64
 2   RoomService                 6954 non-null   int64
 3   Dining                      6954 non-null   int64
 4   Retail                      6954 non-null   int64
 5   Spa                         6954 non-null   int64
 6   Entertainment               6954 non-null   int64
 7   LoyaltyPoints               6954 non-null   int64
 8   SurveyScore                 6954 non-null   int64
 9   DaysSinceEmail              6954 non-null   int64
 10  SharedRoom                  6954 non-null   int64
 11  RoomNumber                  6954 non-null   int64
 12  BookingMonth                6954 non-null   int64
 13  BookingYear                 6954 non-null   int64
 14  PromoCod

In [8]:
#run the test data against the optimized XGBoost model to get the predictions for the submission file

#drop GuestID column from the test data
test_X = test.drop(columns=["GuestID"]) if "GuestID" in test.columns else test.copy()

# Apply the same categorical encoding used during training so feature indices line up.
for c in cat_features:
    test_X[c] = pd.Categorical(
        test_X[c].astype(str).fillna("missing"), categories=category_maps[c]
    ).codes

# Ensure column order matches the training feature order
test_X = test_X[X_train.columns]

probs = xgb.predict(test_X)

#display the probs
print(probs)

#submission already has GuestID; attach predictions as "Churned"
submission["Churned"] = probs

submission.info()

#output the csv file with the predictions
submission.to_csv("submission.csv", index=False)

[1 0 0 ... 1 0 0]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   GuestID  1739 non-null   int64
 1   Churned  1739 non-null   int64
dtypes: int64(2)
memory usage: 27.3 KB
